# DeepSeek-Coder 6.7B × NVIDIA OpenCodeInstruct — Colab T4 QLoRA

**Project repository:** https://github.com/Birajkakadiya/deepseek-finetuning  
**Base model:** `deepseek-ai/deepseek-coder-6.7b-instruct`  
**Dataset:** `nvidia/OpenCodeInstruct`  

This notebook runs a **small real 4-bit QLoRA training smoke test** using your GitHub project. It does **not** need a GitHub GPU runner.

**Before running:** choose **Runtime → Change runtime type → T4 GPU → Save** (subject to Colab availability). Use **Runtime → Run all** only for the *small smoke test*: the 60-step run below is disabled by default.

> A Colab GPU instance, model download, and GPU access cannot be provisioned from GitHub alone. A successful notebook run requires you to connect a Colab GPU. Free T4 access is subject to availability and quotas. Do not publish tokens or output checkpoints containing sensitive data.


## 1. Check your actual GPU and disk capacity

The training script deliberately refuses to train on CPU-only runtimes.

In [ ]:
import subprocess, shutil, torch
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "No CUDA GPU. Colab menu: Runtime → Change runtime type → T4 GPU."
gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"GPU: {gpu}; VRAM: {vram_gib:.1f} GiB; free disk: {free_gib:.1f} GiB")
assert vram_gib >= 14, "This project needs at least approximately 14 GiB of GPU VRAM for the T4 preset."
assert free_gib >= 16, "Not enough free disk space for downloading model weights and training outputs."
subprocess.run(["nvidia-smi"], check=True)

## 2. Clone or refresh your GitHub repository

In [ ]:
from pathlib import Path
import subprocess
repo_url = "https://github.com/Birajkakadiya/deepseek-finetuning.git"
repo_dir = Path("/content/deepseek-finetuning")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)
print("Repository ready:", repo_dir)
assert (repo_dir / "train.py").exists()

## 3. Install the project's Hugging Face training libraries

Colab already provides a CUDA-enabled PyTorch installation. This cell installs the project's other pinned dependencies. If an import error recommends a runtime restart, use **Runtime → Restart session**, then rerun from Step 1.

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(repo_dir / "tests"), "-v"], check=True, cwd=str(repo_dir))

## 4. Train: **2 optimizer steps** (smoke test)

Small run settings: 48 training rows, 8 validation rows, 512 tokens, 4-bit NF4, LoRA rank 16, alpha 32. The first model download may take considerable time. This notebook will not execute or evaluate generated code from the dataset.

In [ ]:
import os
smoke_dir = repo_dir / "outputs" / "colab-t4-smoke"
env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
# Optional: HF_TOKEN may be supplied through Colab Secrets, but the public model/dataset usually do not require it.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        env["HF_TOKEN"] = hf_token
except Exception:
    pass
cmd = [sys.executable, str(repo_dir / "train.py"),
       "--preset", "t4", "--max-steps", "2",
       "--max-train-samples", "48", "--max-eval-samples", "8",
       "--max-length", "512", "--output-dir", str(smoke_dir)]
print("Launching CUDA training smoke test...")
subprocess.run(cmd, check=True, cwd=str(repo_dir), env=env)
print("Smoke test complete. Adapter:", smoke_dir / "adapter")

## 5. Inspect metrics and create a training-loss plot

In [ ]:
import json
metrics_path = smoke_dir / "run_metadata.json"
assert metrics_path.exists(), "No metrics found; training has not finished successfully."
report = json.loads(metrics_path.read_text())
print(json.dumps({"gpu": report["gpu"], "train_rows": report["train_rows"],
                  "eval_rows": report["eval_rows"],
                  "train_metrics": report["train_metrics"],
                  "eval_metrics": report["eval_metrics"]}, indent=2))
subprocess.run([sys.executable, str(repo_dir / "plot_loss.py"),
                "--output-dir", str(smoke_dir)], check=True, cwd=str(repo_dir))
from IPython.display import Image, display
display(Image(filename=str(smoke_dir / "loss.png")))

## 6. Save the LoRA adapter to Google Drive (recommended)

You must authorize Google Drive mounting when Colab prompts. This saves only adapter and run metrics, not the 6.7B-parameter base model. Nothing is pushed to GitHub automatically.

In [ ]:
from google.colab import drive
import zipfile
drive.mount("/content/drive")
archive = Path("/content/drive/MyDrive/deepseek-coder-t4-smoke-adapter.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    files = [p for p in (smoke_dir / "adapter").rglob("*") if p.is_file()]
    files += [smoke_dir / name for name in ("run_metadata.json", "trainer_state.json", "loss.png")
              if (smoke_dir / name).exists()]
    for file in files:
        zf.write(file, file.relative_to(smoke_dir))
print("Saved adapter & metrics to:", archive)

## 7. Optional: run a longer 60-step experiment

**Disabled by default.** Change `RUN_LONG_TRAINING` to `True` and run just this cell *after the smoke test succeeds*. On a free Colab T4, training time and runtime availability are unpredictable; check your remaining quota first. The longer run will write to `outputs/colab-t4-60steps`.

In [ ]:
RUN_LONG_TRAINING = False
if RUN_LONG_TRAINING:
    long_dir = repo_dir / "outputs" / "colab-t4-60steps"
    long_cmd = [sys.executable, str(repo_dir / "train.py"),
                "--preset", "t4", "--max-steps", "60",
                "--max-train-samples", "1500", "--max-eval-samples", "100",
                "--max-length", "1024", "--output-dir", str(long_dir)]
    subprocess.run(long_cmd, check=True, cwd=str(repo_dir), env=env)
    print("Longer experiment finished:", long_dir)
else:
    print("Long run skipped. Set RUN_LONG_TRAINING=True when ready.")

## 8. Optional: generate a coding answer using your smoke-test adapter

This cell re-downloads/loads the model for inference if necessary; on a T4 the adapter must be loaded in 4-bit mode.

In [ ]:
RUN_INFERENCE = False
if RUN_INFERENCE:
    subprocess.run([sys.executable, str(repo_dir / "generate.py"),
                    "--adapter-dir", str(smoke_dir / "adapter"),
                    "--load-in-4bit",
                    "--prompt", "Write a Python function to find prime numbers, with unit tests."],
                   check=True, cwd=str(repo_dir), env=env)
else:
    print("Inference skipped; set RUN_INFERENCE=True after training.")

### Troubleshooting

- **No GPU / CUDA error:** choose a T4 GPU under **Runtime → Change runtime type**, then reconnect.
- **GPU out of memory:** restart the runtime, rerun cells from the top and keep the smoke-test 512-token setting; reduce `--max-length` to 256 if needed.
- **Download/rate limit error:** add an optional Hugging Face access token in Colab **Secrets** under the name `HF_TOKEN`; do **not** paste it into a notebook cell or GitHub.
- **Disconnected session:** reconnect; local Colab files may be lost. Save the adapter to Google Drive after successful training.
- **Training fails after dependency install:** restart the runtime and rerun the cells from Step 1.

See the project [README](https://github.com/Birajkakadiya/deepseek-finetuning/blob/main/README.md) for the A100 and AWS G4dn presets.